# ilo-redfish-emulator
詳細は [ilo-redfish-emulator](https://github.com/HewlettPackard/ilo-redfish-emulator)を参照

## インポート

In [ ]:
import os
import subprocess
from pathlib import Path

## リポジトリをクローンする

In [ ]:
repo_url = "https://github.com/HewlettPackard/ilo-redfish-emulator.git"
target_dir = "ilo-redfish-emulator"
if not os.path.exists(target_dir):
    !git clone {repo_url}
    %cd {target_dir}
else:
    %cd {target_dir}
    !git pull

In [ ]:
os.listdir()

## .envファイル設定
モックアップフォルダやポートを指定する

In [ ]:
env_path = Path(".env")
env_path.write_text(
    "MOCKUP_FOLDER=DL380a_Gen12\n"
    "EXTERNAL_PORT=8443\n"
    "ASYNC_SLEEP=0\n",
    encoding="utf-8"
)
print(env_path.read_text(encoding="utf-8"))

## ビルド

In [ ]:
!podman build -t ilo-emulator:latest .

## 起動

In [ ]:
env = os.environ.copy()
with open(".env") as f:
    for line in f:
        if "=" in line:
            key, value = line.strip().split("=", 1)
            env[key] = value

!podman compose up -d

In [ ]:
!podman compose ps

## 接続確認

### 簡易 (curl)

In [ ]:
!curl -k https://localhost:8443/redfish/v1/

### python-ilorest-library
詳細は [python-ilorest-library](https://github.com/HewlettPackard/python-ilorest-library) を参照

In [ ]:
%pip install python-ilorest-library

In [ ]:
import json
try:
    from redfish.rest_client import redfish_client
except ImportError:
    try:
        from redfish import redfish_client
    except ImportError:
        from redfish import RedfishClient as redfish_client
print(f"Using client: {redfish_client.__name__}")

In [ ]:
ilo = {
    "base_url": "https://127.0.0.1:8443",
    "username": "root",
    "password": "root_password",
}

エミュレーター環境では auth="session" ではなく auth="basic" での接続が必要

In [ ]:
client = redfish_client(**ilo)
client.login(auth="basic")

In [ ]:
res = client.get("/redfish/v1/")
print(json.dumps(res.dict, indent=2, ensure_ascii=False))

In [ ]:
client.logout()

## 停止

In [ ]:
!podman compose down